# 07 — Can Carver size the TEMA book?

Two systems, one overlay.

* **TEMA** decides *when* (event-driven 4h, frozen 9/90/199, isolated 10×, SL/TP).
* **Carver** decides *how much* (continuous forecast → vol-targeted weight, `exec_lag=1`).

Carver does **not** change periods, SL, or TP. Entries/exits stay the frozen list unless we explicitly **filter** (skip ticket if lagged weight < 0.05).

## Honest size

BTC-only Carver mean weight is ~12%. If we set `stake *= weight` raw, DD shrinks because the book got smaller — that is not an overlay result. **Scale reference = mean lagged weight at IS entries**, so OOS average stake ≈ binary TEMA. Then we compare paths.

| Book | What it tests |
|---|---|
| binary | constant $100 isolated stake |
| carver_daily | daily Carver (no CS) as-of onto 4h entries, IS-normalized |
| carver_4h | same-timescale Carver on 4h bars (`ann=2190`) |
| inv_vol | always-long vol target (forecast pinned +10) |
| carver_filter | skip entry if daily held < 0.05 (changes the list) |

## H11

Carver can size TEMA tickets and tighten OOS DD vs binary. The *forecast* vs inverse-vol is the tell: if daily-Carver ≈ inv-vol, you bought a vol dial, not Strat 17–19 skill.

Forecast skill at entry: `corr(fc_t, trade.ret)` — ~0 means no timing alpha on this ticket list.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
import numpy as np
import pandas as pd
from research.trend_lab.carver import VOL_TARGET
from research.trend_lab.data import load_symbol
from research.trend_lab.evaluate import eval_tema
from research.trend_lab.metrics import kpi_table
from research.trend_lab.plots import allocation_fig, equity_overlay, rolling_sharpe_fig, underwater
from research.trend_lab.protocol import split_frame
from research.trend_lab.tema_carver import overlay_pack
from research.trend_lab.tema_robust import daily_kpis
from research.trend_lab.tema_system import TemaParams, daily_equity, tema_bar_equity

btc, src = load_symbol("BTCUSDT", "4h")
parts = split_frame(btc)
p10 = TemaParams(leverage=10.0)
t10 = eval_tema(btc, p10)
print("OOS trades", len(t10["oos_trades"]), "IS trades", len(t10["is_trades"]))

pack = overlay_pack(
    btc, t10["oos_trades"], t10["is_trades"]["entry_time"],
    base_stake=p10.stake, leverage=p10.leverage, cost_bps=p10.cost_bps, vol_target=VOL_TARGET,
)
pack_is = overlay_pack(
    btc, t10["is_trades"], t10["is_trades"]["entry_time"],
    base_stake=p10.stake, leverage=p10.leverage, cost_bps=p10.cost_bps, vol_target=VOL_TARGET,
)
print("IS scale refs", pack["refs"], "FDM", pack["fdm"])

oos_idx, is_idx = parts["oos"].index, parts["is"].index
rows, eqs = {}, {}
for name in ("binary", "carver_daily", "carver_4h", "inv_vol", "carver_filter"):
    rows[f"{name}_OOS"] = daily_kpis(oos_idx, pack[name])
    rows[f"{name}_IS"] = daily_kpis(is_idx, pack_is[name])
    eqs[name] = daily_equity(tema_bar_equity(oos_idx, pack[name])["equity"])
display(kpi_table(rows).round(3))


In [ ]:
equity_overlay(eqs, "OOS TEMA — binary vs Carver size (IS-normalized)").show()
rolling_sharpe_fig({k: v.pct_change().fillna(0) for k, v in eqs.items() if len(v)}, 90, "OOS 90d rolling Sharpe").show()
underwater(eqs["binary"], "OOS binary TEMA DD").show()
underwater(eqs["carver_daily"], "OOS Carver-daily size DD").show()
underwater(eqs["inv_vol"], "OOS inverse-vol size DD").show()
allocation_fig({
    "daily held (lagged)": pack["held_daily"].reindex(oos_idx),
    "4h held (lagged)": pack["held_4h"].reindex(oos_idx),
}, "OOS Carver weight on the 4h index").show()

# forecast skill on the frozen OOS ticket list
fcs, rets = [], []
fc = pack["fc_daily"].sort_index()
for _, r in t10["oos_trades"].iterrows():
    v = fc.asof(r["entry_time"])
    if v is not None and np.isfinite(v) and not pd.isna(v):
        fcs.append(float(v)); rets.append(float(r["ret"]))
if len(fcs) >= 8:
    corr = float(np.corrcoef(fcs, rets)[0, 1])
    hit = float(np.mean((np.array(fcs) > 0) == (np.array(rets) > 0)))
    print(f"OOS corr(fc, trade ret)={corr:.3f}  hit={hit:.3f}  n={len(fcs)}")
else:
    print("not enough overlapping forecasts")
print("mean OOS daily held", float(pack["held_daily"].reindex(oos_idx).mean()))
print("median OOS daily held", float(pack["held_daily"].reindex(oos_idx).median()))


## How to implement (research → desk, still no broker)

1. Keep TEMA entries exactly as live (9/90/199, ADX/ATR/RSI gates, SL/TP).
2. At signal close, read **yesterday’s** Carver weight (daily last 4h close, `exec_lag=1`).
3. `stake_eff = stake_ref * clip(w / w_IS_mean, 0, 2.5)`. Isolated cap on `stake_eff`.
4. Do **not** skip the ticket from a weak forecast unless H11’s filter book clearly wins OOS *and* DF — that is a second change.
5. Do **not** write Carver into `W_*` or Pine. Sizing is not scoring.

If corr(fc, ret) ≈ 0 and Carver-daily ≈ inv-vol, ship a **vol dial** (smaller tickets in high vol), not a forecast engine.
